In [1]:
import pandas as pd 
import os 
usuario = os.getlogin()

In [2]:
plantillas = pd.read_parquet (fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Comando Florence Nightingale\Proyectos\5_Censo\Anterior\Analisis Brecha\Data\Plantilla ideal\Plantilla ideal minima\plantillas_ideales_minimas.parquet")
ocupacion = pd.read_excel (fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\Plantilla\ocupacion.xlsx")    
brecha = pd.read_excel (fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\Plantilla\brecha_minima.xlsx")
clues = pd.read_parquet (fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")  
puesto = pd.read_excel (fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Comando Florence Nightingale\Proyectos\5_Censo\Anterior\Estados duplicados e inconsistencias\Catalogos\Catálogo de Código de Puesto CRH.xlsx",skiprows=1)   


In [3]:
puesto.columns = puesto.columns.str.lower().str.replace(' ', '_')

In [6]:
from flask import Flask, render_template, request, jsonify, send_from_directory
import pandas as pd
import os
import re
from werkzeug.utils import secure_filename
from datetime import datetime
import json

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = 'uploads'
app.config['MAX_CONTENT_LENGTH'] = 5 * 1024 * 1024  # 5MB
app.config['SECRET_KEY'] = 'tu-clave-secreta-aqui'

# Crear carpeta de uploads si no existe
os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)

# ============================================
# CARGA DE DATOS
# ============================================

def cargar_datos():
    """Carga los datos desde los archivos Parquet y Excel"""
    try:
        # Ruta base - AJUSTA ESTA RUTA SEGÚN TU ESTRUCTURA
        ruta_base = r"C:\Users\tu_usuario\IMSS-BIENESTAR"
        
        # Cargar CLUES
        ruta_clues = os.path.join(ruta_base, "División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")
        clues_df = pd.read_parquet(ruta_clues)
        
        # Cargar Puestos
        ruta_puestos = os.path.join(ruta_base, "División de Procesamiento de información - Comando Florence Nightingale\Proyectos\5_Censo\Anterior\Estados duplicados e inconsistencias\Catalogos\Catálogo de Código de Puesto CRH.xlsx")
        puestos_df = pd.read_excel(ruta_puestos, skiprows=1)
        
        return clues_df, puestos_df
    
    except Exception as e:
        print(f"Error al cargar datos: {e}")
        # Retornar DataFrames vacíos como fallback
        return pd.DataFrame(), pd.DataFrame()

# Cargar datos al iniciar
clues_df, puestos_df = cargar_datos()

# ============================================
# FUNCIONES DE VALIDACIÓN
# ============================================

def validar_curp(curp):
    """Validación R01 y R02 - Formato y dígito verificador de CURP"""
    if not curp or len(curp) != 18:
        return {
            'valido': False,
            'mensaje': 'La CURP debe tener 18 caracteres'
        }
    
    curp = curp.upper()
    
    # R01: Validar formato oficial
    patron = r'^[A-Z]{4}[0-9]{6}[A-Z]{6}[0-9]{2}$'
    if not re.match(patron, curp):
        return {
            'valido': False,
            'mensaje': 'Formato de CURP inválido. Ejemplo: GODE561231HDFRRL09'
        }
    
    # R02: Validar dígito verificador (algoritmo completo)
    try:
        # Diccionario de caracteres para el algoritmo
        caracteres = '0123456789ABCDEFGHIJKLMNÑOPQRSTUVWXYZ'
        valores = {c: i for i, c in enumerate(caracteres)}
        
        # Calcular dígito verificador
        suma = 0
        for i, char in enumerate(curp[:17]):
            valor = valores.get(char, 0)
            # Aplicar factor según posición (primero 14, luego 13, etc.)
            factor = 14 - (i % 10)
            if factor < 0:
                factor = 0
            suma += valor * factor
        
        digito_esperado = (10 - (suma % 10)) % 10
        digito_obtenido = int(curp[17]) if curp[17].isdigit() else -1
        
        if digito_obtenido == digito_esperado:
            return {
                'valido': True,
                'mensaje': 'CURP válida'
            }
        else:
            return {
                'valido': False,
                'mensaje': 'Dígito verificador incorrecto'
            }
    
    except Exception as e:
        return {
            'valido': False,
            'mensaje': f'Error en validación: {str(e)}'
        }

def validar_puesto(puesto):
    """Validación R03 - El puesto existe en el catálogo"""
    if not puesto:
        return {
            'valido': False,
            'mensaje': 'Debe seleccionar un puesto'
        }
    
    existe = len(puestos_df[puestos_df['descripcion_de_puesto'] == puesto]) > 0
    return {
        'valido': existe,
        'mensaje': 'Puesto válido' if existe else 'Puesto no encontrado en catálogo'
    }

def validar_clues(clues):
    """Validación R04 - La CLUES existe en el catálogo"""
    if not clues:
        return {
            'valido': False,
            'mensaje': 'Debe ingresar una CLUES'
        }
    
    existe = len(clues_df[clues_df['clues_imb'] == clues.upper()]) > 0
    return {
        'valido': existe,
        'mensaje': 'CLUES válida' if existe else 'CLUES no encontrada en catálogo'
    }

# ============================================
# RUTAS DE LA API
# ============================================

@app.route('/')
def index():
    """Página principal con el formulario"""
    return render_template('index.html')

@app.route('/api/search_clues')
def search_clues():
    """Buscar CLUES para autocompletado"""
    query = request.args.get('q', '').strip()
    
    if len(query) < 2:
        return jsonify([])
    
    # Buscar en CLUES y nombre de unidad
    resultados = clues_df[
        clues_df['clues_imb'].str.contains(query, case=False, na=False) |
        clues_df['nombre_de_la_unidad'].str.contains(query, case=False, na=False)
    ]
    
    # Limitar y formatear resultados
    resultados = resultados.head(20)
    
    return jsonify(resultados.to_dict('records'))

@app.route('/api/get_clues_data')
def get_clues_data():
    """Obtener datos completos de una CLUES específica"""
    clues_id = request.args.get('clues', '').upper()
    
    resultado = clues_df[clues_df['clues_imb'] == clues_id]
    
    if len(resultado) > 0:
        row = resultado.iloc[0]
        return jsonify({
            'nombre': row.get('nombre_de_la_unidad', ''),
            'entidad': row.get('entidad', ''),
            'municipio': row.get('municipio', '')
        })
    
    return jsonify({})

@app.route('/api/get_puestos')
def get_puestos():
    """Obtener lista de puestos"""
    # Limpiar datos y eliminar duplicados
    puestos_limpios = puestos_df.drop_duplicates(subset=['descripcion_de_puesto'])
    
    # Asegurar que existan las columnas necesarias
    if 'nivel' not in puestos_limpios.columns:
        puestos_limpios['nivel'] = ''
    
    return jsonify(puestos_limpios.to_dict('records'))

@app.route('/api/validate_curp', methods=['POST'])
def validate_curp():
    """Endpoint para validar CURP"""
    data = request.get_json()
    curp = data.get('curp', '').upper()
    resultado = validar_curp(curp)
    return jsonify(resultado)

@app.route('/api/validate_puesto', methods=['POST'])
def validate_puesto():
    """Endpoint para validar puesto"""
    data = request.get_json()
    puesto = data.get('puesto', '')
    resultado = validar_puesto(puesto)
    return jsonify(resultado)

@app.route('/api/validate_clues', methods=['POST'])
def validate_clues():
    """Endpoint para validar CLUES"""
    data = request.get_json()
    clues = data.get('clues', '')
    resultado = validar_clues(clues)
    return jsonify(resultado)

@app.route('/api/upload_document', methods=['POST'])
def upload_document():
    """Subir documento PDF"""
    if 'file' not in request.files:
        return jsonify({'error': 'No se envió ningún archivo'}), 400
    
    file = request.files['file']
    
    if file.filename == '':
        return jsonify({'error': 'No se seleccionó ningún archivo'}), 400
    
    # Validar tipo de archivo
    if not file.filename.lower().endswith('.pdf'):
        return jsonify({'error': 'Solo se permiten archivos PDF'}), 400
    
    # Validar tamaño
    file.seek(0, 2)
    size = file.tell()
    file.seek(0)
    
    if size > 5 * 1024 * 1024:
        return jsonify({'error': 'El archivo excede el límite de 5MB'}), 400
    
    # Guardar archivo
    filename = secure_filename(file.filename)
    # Agregar timestamp para evitar duplicados
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    nombre_guardado = f"{timestamp}_{filename}"
    ruta_guardado = os.path.join(app.config['UPLOAD_FOLDER'], nombre_guardado)
    file.save(ruta_guardado)
    
    return jsonify({
        'success': True,
        'filename': filename,
        'saved_name': nombre_guardado,
        'message': 'Archivo subido correctamente'
    })

@app.route('/api/save_candidate', methods=['POST'])
def save_candidate():
    """Guardar datos del candidato (simulado)"""
    data = request.get_json()
    
    # Aquí puedes guardar en base de datos o archivo
    # Por ahora solo validamos y respondemos
    
    # Validaciones adicionales antes de guardar
    curp_validacion = validar_curp(data.get('curp', ''))
    if not curp_validacion['valido']:
        return jsonify({
            'success': False,
            'error': curp_validacion['mensaje']
        }), 400
    
    # Generar un ID único para el candidato
    candidato_id = datetime.now().strftime('%Y%m%d%H%M%S')
    
    # Guardar en un archivo JSON (opcional)
    try:
        # Crear carpeta de candidatos si no existe
        os.makedirs('candidatos', exist_ok=True)
        
        # Guardar datos
        archivo = os.path.join('candidatos', f'candidato_{candidato_id}.json')
        with open(archivo, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        
        return jsonify({
            'success': True,
            'candidato_id': candidato_id,
            'message': 'Candidato guardado correctamente'
        })
    
    except Exception as e:
        return jsonify({
            'success': False,
            'error': f'Error al guardar: {str(e)}'
        }), 500

# ============================================
# RUTAS PARA ARCHIVOS ESTÁTICOS
# ============================================

@app.route('/uploads/<filename>')
def uploaded_file(filename):
    """Servir archivos subidos"""
    return send_from_directory(app.config['UPLOAD_FOLDER'], filename)

# ============================================
# INICIAR APLICACIÓN
# ============================================



Error al cargar datos: [Errno 2] No such file or directory: 'C:\\Users\\tu_usuario\\IMSS-BIENESTAR\\División de Procesamiento de información - Repositorio de Datos\\CLUES\\clues.parquet'


<>:28: SyntaxWarning: invalid escape sequence '\C'
<>:32: SyntaxWarning: invalid escape sequence '\P'
<>:28: SyntaxWarning: invalid escape sequence '\C'
<>:32: SyntaxWarning: invalid escape sequence '\P'
C:\Users\jose.valdez\AppData\Local\Temp\ipykernel_30440\1054057534.py:28: SyntaxWarning: invalid escape sequence '\C'
  ruta_clues = os.path.join(ruta_base, "División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")
C:\Users\jose.valdez\AppData\Local\Temp\ipykernel_30440\1054057534.py:32: SyntaxWarning: invalid escape sequence '\P'
  ruta_puestos = os.path.join(ruta_base, "División de Procesamiento de información - Comando Florence Nightingale\Proyectos\5_Censo\Anterior\Estados duplicados e inconsistencias\Catalogos\Catálogo de Código de Puesto CRH.xlsx")
